# Chapter 7: The Full VLA
Every piece from Chapters 2-6, assembled into one SmolVLA-like model: **frozen SigLIP vision + frozen SmolLM2 language + a trainable flow-matching action expert**, trained on real SO100 robot data.

This notebook builds the model, inspects it, and runs a full forward pass and sampling loop. Training on the real 78K-frame dataset needs a 24 GB GPU and ~3.2 hours -- that command is given at the end, along with the published results.

In [ ]:
!pip install torch torchvision numpy matplotlib transformers pillow

In [ ]:
# Skips the clone if it is already present, and surfaces the real error if
# it fails, rather than hiding it and failing confusingly on the %cd below.
![ -d vla-from-scratch ] || git clone https://github.com/FanFeast/vla-from-scratch.git
%cd vla-from-scratch/chapters/07_full_vla

## The Assembly

```
Image    -> SigLIP (frozen) -> pixel shuffle 4x4 -> connector -> 64 tokens x 960
Language -> SmolLM2 tokenizer ---------------------------------\
State    -> Linear(6 -> 960) -> 1 token -----------------------> prefix
Prefix   -> SmolLM2 (frozen, first 16 of 32 layers) -> hidden states @ L5, L10, L16
Actions  -> Flow-matching action expert (trainable) -> chunk (10 x 6)
```

Two ideas do the heavy lifting:

**Only the action expert trains.** The backbone comes from `SmolVLM2-500M-Video-Instruct` and stays frozen. 20.8M trainable out of 323.7M.

**The language model is truncated at half depth.** Layers 17-32 exist to produce text, and we never generate text -- we read hidden states. Running 16 layers instead of 32 halves the prefix cost for no measured loss.

In [ ]:
import torch
from config import SmolVLAConfig, PRESETS
from model import build_smolvla

preset = PRESETS["so100"]
config = SmolVLAConfig(
    action_dim=preset["action_dim"],
    state_dim=preset["state_dim"],
    chunk_size=preset["chunk_size"],
)
model = build_smolvla(config)

counts = model.count_parameters()
print(f"\nTotal      {counts['total']/1e6:8.1f}M")
print(f"Frozen     {counts['frozen']/1e6:8.1f}M  ({counts['frozen']/counts['total']*100:.1f}%)")
print(f"Trainable  {counts['trainable']/1e6:8.1f}M  ({counts['trainable']/counts['total']*100:.1f}%)")

## Pixel Shuffle: 1024 Tokens -> 64

SigLIP on a 512x512 image gives 1024 patch tokens. Feeding all of them into the language model would dominate the prefix and the compute.

`pixel_shuffle` trades **space for channels**: a 4x4 block of neighbouring patches folds into one token with 16x the channel width. Nothing is discarded -- the information is rearranged, and the sequence gets 16x shorter.

In [ ]:
from model import pixel_shuffle

x = torch.randn(1, 1024, 768)          # (batch, tokens, dim) from SigLIP
y = pixel_shuffle(x, scale_factor=4)

print(f"in : {tuple(x.shape)}   -> {x.shape[1]} tokens x {x.shape[2]}D")
print(f"out: {tuple(y.shape)}   -> {y.shape[1]} tokens x {y.shape[2]}D")
print(f"\nsequence length  /{x.shape[1] // y.shape[1]}")
print(f"channel width    x{y.shape[2] // x.shape[2]}")
print(f"total values preserved: {x.numel() == y.numel()}")

## A Forward Pass

Synthetic inputs of the right shapes, so this runs without the dataset. The expert takes noisy actions plus a flow-matching timestep and predicts a **velocity** -- the direction from noise toward the true action chunk.

In [ ]:
B, K, A = 2, config.chunk_size, config.action_dim
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()

vision_tokens = torch.randn(B, config.num_vision_tokens, config.vlm_hidden_dim, device=device)
lang_ids      = torch.randint(0, 1000, (B, 12), device=device)
lang_mask     = torch.ones(B, 12, dtype=torch.long, device=device)
states        = torch.randn(B, config.state_dim, device=device)
noisy_actions = torch.randn(B, K, A, device=device)
timesteps     = torch.rand(B, device=device)

with torch.no_grad():
    velocity = model(vision_tokens, lang_ids, lang_mask, states, noisy_actions, timesteps)

print(f"vision tokens {tuple(vision_tokens.shape)}")
print(f"language      {tuple(lang_ids.shape)}")
print(f"state         {tuple(states.shape)}")
print(f"noisy actions {tuple(noisy_actions.shape)}")
print(f"-> velocity   {tuple(velocity.shape)}   (matches the action chunk)")

## Sampling: Noise -> Action Chunk

Inference integrates the learned velocity field from pure noise to an action chunk with 10 Euler steps. Timesteps during *training* are drawn from `Beta(1.5, 1.0)`, which oversamples the high-noise end where the field is hardest to learn.

In [ ]:
from train import FlowMatchingVLA

fm = FlowMatchingVLA(model, config)
with torch.no_grad():
    chunk = fm.sample(vision_tokens, lang_ids, lang_mask, states, ema=None, num_steps=10)

print(f"sampled chunk: {tuple(chunk.shape)}  ({K} future steps x {A} DOF)")
print(f"range: [{chunk.min():.3f}, {chunk.max():.3f}]  (normalized)")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.5))
for j in range(A):
    ax.plot(chunk[0, :, j].float().cpu(), marker="o", ms=3, label=f"joint {j}")
ax.set_xlabel("Step within chunk"); ax.set_ylabel("Normalized action")
ax.set_title("One sampled action chunk (untrained model -- structure, not skill)")
ax.legend(fontsize=7, ncol=3); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Training on Real Data

The real run uses 158 episodes / 78,300 frames across three SO100 datasets (pick-place, stacking, sorting). It is a two-phase pipeline, and the split matters:

1. **Extraction (~35 min, once):** decode video, run SigLIP + pixel shuffle + connector, cache 9.7 GB of vision tokens. This takes the vision encoder out of the training loop entirely.
2. **Training (~3.2 h):** load cached tokens, run frozen SmolLM2 per batch, train the action expert. ~58 s/epoch.

```bash
uv sync
python train.py --preset so100          # extract + train, 200 epochs
```

Needs ~24 GB VRAM. The cell below is left unexecuted on purpose.

In [ ]:
# Uncomment to run the full pipeline (downloads ~13 GB, needs a 24 GB GPU):
# from train import run_preset
# run_preset("so100")

print("Published results -- SO100, 158 episodes, 200 epochs, RTX 4090, 3.2h\n")
rows = [
    ("Epoch 25",  0.71, 0.20, 3.36, 77.9),
    ("Epoch 50",  0.57, 0.24, 2.98, 69.1),
    ("Epoch 100", 0.89, 0.80, 2.22, 51.4),
    ("Epoch 150", 0.56, 0.48, 2.43, 56.2),
    ("Epoch 200", 0.63, 0.59, 2.41, 55.9),
]
print(f"{'Checkpoint':<12}{'Train':>8}{'Val':>8}{'MAE norm':>10}{'MAE raw':>10}")
print("-" * 48)
for name, tr, va, mn, mr in rows:
    star = "  <- best" if name == "Epoch 100" else ""
    print(f"{name:<12}{tr:>8.2f}{va:>8.2f}{mn:>10.2f}{mr:>10.1f}{star}")

## The Reality Check

Loss curves flatter you. `policy_reality_check.py` replays held-out episodes and plots the policy's predicted chunk against the ground-truth action the human actually commanded -- the offline version of "does this thing work?"

Chapter 5 taught the lesson the expensive way; this is the cheap way to keep checking it.

In [ ]:
from IPython.display import Image, display

display(Image("figures/ch07_policy_reality_check.png"))
display(Image("figures/ch07_training_curves.png"))

# Regenerate from a checkpoint (requires the cached dataset):
# !python policy_reality_check.py --checkpoint checkpoints/so100_epoch100.pt

## What We Learned

**The whole pipeline works end to end** -- raw video and a language instruction in, a 6-DOF action chunk out, trained by flow matching on real robot data.

**Freezing almost everything is the point.** 20.8M of 323.7M params train (6.4%). The pretrained backbone already knows what a cube and a gripper look like; the action expert only has to learn how *this* robot moves.

**Best checkpoint is epoch 100, not 200.** MAE improves 34% from epoch 25 to 100, then drifts as the model starts memorizing 158 episodes. Train loss keeps falling the whole time -- another reminder to select on validation, not training.

**Raw MAE stays high (51.4).** The model learns real structure but is not task-competent, and the reason is data, not architecture: SmolVLA trained on 481 datasets; we used 3. Saying so plainly is more useful than a curve that looks nice.

**Next:** Chapter 8 adapts that frozen backbone with LoRA -- 0.13% of the parameters -- and measures whether it actually helps.